## RetentionAI — 02. Data Understanding

Before any visualization or modeling, this notebook checks what the raw data actually is: the row grain, the target balance, and any dtype or encoding quirks. No feature engineering and no hypothesis testing here — that happens in 03.

Findings from this notebook (the TotalCharges fix, the structural-dependency flag) are implemented once in `src/data_prep.py` and consumed by 03, not re-derived there.

In [3]:
import pandas as pd
from src.config import RAW_CSV_PATH

df = pd.read_csv(RAW_CSV_PATH)
df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


---

### 1. Grain

One row should equal one customer, identified by `customerID`. Checking for duplicates before doing anything else — if this is wrong, everything downstream is wrong.

In [4]:
print(f"Total rows: {df.shape[0]}")
print(f"Unique customer IDs: {df['customerID'].nunique()}")
print(f"Duplicate customer IDs: {df['customerID'].duplicated().sum()}")

assert df['customerID'].nunique() == len(df), "customerID is not unique — grain assumption is wrong."
print("\nGrain confirmed: one row = one customer.")

Total rows: 7043
Unique customer IDs: 7043
Duplicate customer IDs: 0

Grain confirmed: one row = one customer.


### 2. Target balance (Churn)

Checking the class split before anything else, since a heavy imbalance changes how the model needs to be evaluated and trained.

In [5]:
print("Absolute counts:")
print(df['Churn'].value_counts())
print("\nProportions:")
print(df['Churn'].value_counts(normalize=True).round(4))

Absolute counts:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Proportions:
Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64


### 3. Data types

In [6]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

`TotalCharges` is `object` instead of numeric. Before coercing it with `errors='coerce'`, checking whether the non-numeric values are blank strings rather than proper `NaN` — pandas treats these differently.

In [7]:
blank_mask = df['TotalCharges'].str.strip() == ''
print(f"Blank TotalCharges entries: {blank_mask.sum()}")
print("\ntenure for those rows:")
print(df.loc[blank_mask, 'tenure'].value_counts())

Blank TotalCharges entries: 11

tenure for those rows:
tenure
0    11
Name: count, dtype: int64


**Finding:** 11 rows have `TotalCharges` as a blank string, and all 11 have `tenure == 0`. These are new customers who haven't reached a first billing cycle yet, so 0 is the correct fill value — not a dropped row, not an average.

**Fix:** coerce to numeric, then fill the resulting `NaN` with 0. Implemented in `src/data_prep.clean_totalcharges()` and applied in `03_eda_cleaning.ipynb`.

### 4. Structural dependencies

Columns like `OnlineSecurity` and `StreamingTV` have three values: `Yes`, `No`, and `No internet service`. That third value isn't a missing value — it's a fact that follows from `InternetService == 'No'`. Replacing it with a plain `No` would erase that distinction.

Checking this holds for all six add-on columns, not just one, since the same claim gets used later as a multicollinearity flag.

In [8]:
addon_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
]

for col in addon_cols:
    print(f"--- InternetService vs {col} ---")
    display(pd.crosstab(df['InternetService'], df[col]))

print("--- PhoneService vs MultipleLines ---")
display(pd.crosstab(df['PhoneService'], df['MultipleLines']))

--- InternetService vs OnlineSecurity ---


OnlineSecurity,No,No internet service,Yes
InternetService,,,
DSL,1241,0,1180
Fiber optic,2257,0,839
No,0,1526,0


--- InternetService vs OnlineBackup ---


OnlineBackup,No,No internet service,Yes
InternetService,,,
DSL,1335,0,1086
Fiber optic,1753,0,1343
No,0,1526,0


--- InternetService vs DeviceProtection ---


...

StreamingTV,No,No internet service,Yes
InternetService,,,
DSL,1464,0,957
Fiber optic,1346,0,1750
No,0,1526,0


--- InternetService vs StreamingMovies ---


StreamingMovies,No,No internet service,Yes
InternetService,,,
DSL,1440,0,981
Fiber optic,1345,0,1751
No,0,1526,0


--- PhoneService vs MultipleLines ---


MultipleLines,No,No phone service,Yes
PhoneService,,,
No,0,682,0
Yes,3390,0,2971


**Finding:** for all six columns, `No internet service` appears only when `InternetService == 'No'`, and never otherwise. Same pattern for `MultipleLines` against `PhoneService`. These six columns move together — flagging this for multicollinearity checks (VIF) in the modeling phase.

### 5. SeniorCitizen encoding

In [9]:
print(f"SeniorCitizen unique values: {sorted(df['SeniorCitizen'].unique())}")
print(f"Partner unique values:       {sorted(df['Partner'].unique())}")

SeniorCitizen unique values: [np.int64(0), np.int64(1)]
Partner unique values:       ['No', 'Yes']


`SeniorCitizen` is stored as `0/1`; `Partner` is stored as `Yes/No`. Noting the inconsistency here — encoding it is a preprocessing step, not a data-understanding one, so it's left as-is for now.

Full feature-by-feature reference is in the project [README](../README.md).